In [ ]:
!pip install scanpy squidpy

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [4]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention")
cwd = os.getcwd()
print(cwd)
sys.path.append("../../")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention


In [5]:
import scanpy as sc
import squidpy as sq
import pandas as pd
from tqdm.notebook import tqdm
import scipy as sp
import numpy as np

import matplotlib.pyplot as plt

In [6]:
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'DejaVu Sans'

pltkw = dict(bbox_inches='tight', transparent=True)

In [7]:
import HadmardAttention as HA

In [8]:
import importlib
import HadmardAttention.tools
import HadmardAttention.model

## Data

In [91]:
adata = sc.read_h5ad("../../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

Founsation Models

In [92]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [64]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/h_optimus.h5ad")
M = edata.obsm['h_optimus']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/virchow.h5ad")
M = edata.obsm['virchow']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [93]:
edata.obs_names = [cid[61:] for cid in edata.obs['cell_id']]
common_cells = ad.obs_names.intersection(edata.obs_names)
ad.obsm['morpho'][ad.obs_names.get_indexer(common_cells)] = \
  edata.obsm['UNI'][edata.obs_names.get_indexer(common_cells)]

In [94]:
#normalizer = Normalizer(norm="l2")
#ad.obsm['morpho'] = normalizer.transform(ad.obsm['morpho'])

adata.obsm['Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

In [95]:
from sklearn.decomposition import PCA

pca = PCA(n_components=500)
X_reduced = pca.fit_transform(adata.obsm['Morpho_Embedding'])
adata.obsm['p_Morpho_Embedding'] = X_reduced

If saved before:

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/rFMs_3/virchow_adata.h5ad")

Noise

In [54]:
rng = np.random.default_rng(42)
ad = adata.copy()
ad.obsm["morpho"] = rng.standard_normal((63173, 500))

In [55]:
adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

Preparing Dataset

In [ ]:
importlib.reload(HA.dataset)
importlib.reload(HA.model)
importlib.reload(HA)

<module 'HadmardAttention' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention/../../HadmardAttention/__init__.py'>

In [96]:
adata = HA.prep_adatas(adata, norm=True, log1p=True)
dataset = HA.make_dataset(adata, sparse_graph=True)

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention/../../HadmardAttention/dataset.py:37: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(adata)


INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        
Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.


In [57]:
print('Expression', dataset[0][0].shape)
print('Morpho_Embedding', dataset[0][1].shape)
print('Neighborhood_Graph', dataset[0][2].shape)

Expression torch.Size([63173, 2000])
Morpho_Embedding torch.Size([63173, 500])
Neighborhood_Graph torch.Size([2, 505384])


In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

## Reconstruction Error

In [100]:
model_type = 5

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


### virchow

In [ ]:
if device == 'cpu':
  model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_virchow_{model_type}.pth',weights_only=True, map_location=torch.device('cpu')))
else:
  model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_virchow_{model_type}.pth',weights_only=True, map_location=torch.device('cpu')))

In [ ]:
## UNI
loader = DataLoader(dataset, batch_size=1, shuffle=False)
criterion = nn.MSELoss(reduction='mean')

with torch.no_grad():
  for x, m, adj_list in loader:

    adj_list = adj_list.squeeze(0).to(device)
    x = x.squeeze(0).to(device)
    m = m.squeeze(0).to(device)

    res = model(adj_list, x, m, x, m, get_details=False)
    loss = criterion(res, x)
    print(loss)

model_type:  0
tensor(0.0990)


In [ ]:
## UNI
loader = DataLoader(dataset, batch_size=1, shuffle=False)
criterion = nn.MSELoss(reduction='mean')

with torch.no_grad():
  for x, m, adj_list in loader:

    adj_list = adj_list.squeeze(0).to(device)
    x = x.squeeze(0).to(device)
    m = m.squeeze(0).to(device)

    res = model(adj_list, x, m, x, m, get_details=False)
    loss = criterion(res, x)
    print(loss)

model_type:  5
tensor(0.0987)


### UNI

In [101]:
if device == 'cpu':
  model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_UNI_{model_type}.pth',weights_only=True, map_location=torch.device('cpu')))
else:
  model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_UNI_{model_type}.pth',weights_only=True, map_location=torch.device('cpu')))

In [99]:
## UNI
loader = DataLoader(dataset, batch_size=1, shuffle=False)
criterion = nn.MSELoss(reduction='mean')

with torch.no_grad():
  for x, m, adj_list in loader:

    adj_list = adj_list.squeeze(0).to(device)
    x = x.squeeze(0).to(device)
    m = m.squeeze(0).to(device)

    masked_x = model.masking(x, 0)

    res = model(adj_list, x, m, masked_x, m, get_details=False)
    loss = criterion(res, x)
    print(loss)

model_type:  0
tensor(0.0531, device='cuda:0')


In [102]:
## UNI
loader = DataLoader(dataset, batch_size=1, shuffle=False)
criterion = nn.MSELoss(reduction='mean')

with torch.no_grad():
  for x, m, adj_list in loader:

    adj_list = adj_list.squeeze(0).to(device)
    x = x.squeeze(0).to(device)
    m = m.squeeze(0).to(device)

    masked_x = model.masking(x, 0)

    res = model(adj_list, x, m, masked_x, m, get_details=False)
    loss = criterion(res, x)
    print(loss)

model_type:  5
tensor(0.0535, device='cuda:0')


### h_optimus

In [88]:
if device == 'cpu':
  model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_hoptimus_{model_type}.pth',weights_only=True, map_location=torch.device('cpu')))
else:
  model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_hoptimus_{model_type}.pth',weights_only=True))

In [86]:
## h_optimus
loader = DataLoader(dataset, batch_size=1, shuffle=False)
criterion = nn.MSELoss(reduction='mean')

with torch.no_grad():
  for x, m, adj_list in loader:

    adj_list = adj_list.squeeze(0).to(device)
    x = x.squeeze(0).to(device)
    m = m.squeeze(0).to(device)

    masked_x = model.masking(x, 0)

    res = model(adj_list, x, m, masked_x, m, get_details=False)
    loss = criterion(res, x)
    print(loss)

model_type:  0
tensor(0.1086, device='cuda:0')


In [89]:
## h_optimus
loader = DataLoader(dataset, batch_size=1, shuffle=False)
criterion = nn.MSELoss(reduction='mean')

with torch.no_grad():
  for x, m, adj_list in loader:

    adj_list = adj_list.squeeze(0).to(device)
    x = x.squeeze(0).to(device)
    m = m.squeeze(0).to(device)

    masked_x = model.masking(x, 0)

    res = model(adj_list, x, m, masked_x, m, get_details=False)
    loss = criterion(res, x)
    print(loss)

model_type:  5
tensor(0.0644, device='cuda:0')


### NOISE

In [61]:
if device == 'cpu':
  model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_NOISE_{model_type}.pth',weights_only=True, map_location=torch.device('cpu')))
else:
  model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_NOISE_{model_type}.pth',weights_only=True))

In [ ]:
## NOISE
loader = DataLoader(dataset, batch_size=1, shuffle=False)
criterion = nn.MSELoss(reduction='mean')

with torch.no_grad():
  for x, m, adj_list in loader:

    adj_list = adj_list.squeeze(0).to(device)
    x = x.squeeze(0).to(device)
    m = m.squeeze(0).to(device)

    masked_x = model.masking(x, 0)

    res = model(adj_list, x, m, masked_x, m, get_details=False)
    loss = criterion(res, x)
    print(loss)

In [62]:
## NOISE
loader = DataLoader(dataset, batch_size=1, shuffle=False)
criterion = nn.MSELoss(reduction='mean')

with torch.no_grad():
  for x, m, adj_list in loader:

    adj_list = adj_list.squeeze(0).to(device)
    x = x.squeeze(0).to(device)
    m = m.squeeze(0).to(device)

    masked_x = model.masking(x, 0)

    res = model(adj_list, x, m, masked_x, m, get_details=False)
    loss = criterion(res, x)
    print(loss)

model_type:  5
tensor(0.0587, device='cuda:0')


## Gene Vector Analysis

## ROI Segmentation

## Cell Type Clustering

## Embedding Similarity